# Transport SSM Hyperparameter Study

This notebook runs Optuna studies for **S4** and **S4D** on the transport dataset.

- Optimization target: validation POD-mode MSE
- Also reported: field reconstruction RMSRE (%)
- Tracking backend: MLflow (all trial params + metrics)
- Includes a smoke test before full search

In [1]:
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import optuna
import mlflow

# Keep notebook output clean during long studies.
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='IProgress not found')
warnings.filterwarnings('ignore', message='.*TqdmExperimentalWarning.*')
warnings.filterwarnings('ignore', message='.*distutils.*')
warnings.filterwarnings('ignore', module='mlflow')
warnings.filterwarnings('ignore', module='optuna')
optuna.logging.set_verbosity(optuna.logging.WARNING)

project_root = Path('..').resolve()
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'utils'))

from transport_data import generate_trajectories, compute_pod

np.random.seed(0)
torch.manual_seed(0)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Optuna:', optuna.__version__)

/Users/markussandnes/Desktop/NCD-ROM-PI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
Optuna: 4.7.0


In [ ]:
Nx = 128
Nt = 400
T = 10.0
n_train = 100
n_test = 20 
n_total = n_train + n_test

x = np.linspace(0, 2 * np.pi, Nx, endpoint=False)
t_grid = np.linspace(0, T, Nt)

all_snaps, all_ic, speeds = generate_trajectories(
    n_total=n_total, Nx=Nx, Nt=Nt, T=T, c_range=(1.0, 3.5), seed=42
)

snaps_train = all_snaps[:n_train]
snaps_test = all_snaps[n_train:]

Phi, S, S_full = compute_pod(snaps_train, n_svd=100, energy_threshold=0.999)
n_modes = Phi.shape[1]

modes_train = snaps_train @ Phi
modes_test = snaps_test @ Phi

n_sensors = 5
sensor_pos = [int((i + 1) * Nx / (n_sensors + 1)) for i in range(n_sensors)]

scaler = StandardScaler()
scaler.fit(modes_train.reshape(-1, n_modes))
A_train_sc = scaler.transform(modes_train.reshape(-1, n_modes)).reshape(n_train, Nt, n_modes)
A_test_sc = scaler.transform(modes_test.reshape(-1, n_modes)).reshape(n_test, Nt, n_modes)

X_all = all_snaps[:, :, sensor_pos]

val_frac = 0.15
n_val_traj = max(1, int(n_train * val_frac))
perm = np.random.permutation(n_train)
tr_idx = perm[n_val_traj:]
vl_idx = perm[:n_val_traj]

X_tr = X_all[:n_train][tr_idx].astype(np.float32)
Y_tr = A_train_sc[tr_idx].astype(np.float32)
X_vl = X_all[:n_train][vl_idx].astype(np.float32)
Y_vl = A_train_sc[vl_idx].astype(np.float32)

print('n_modes:', n_modes)
print('X_tr:', X_tr.shape, 'Y_tr:', Y_tr.shape)
print('X_vl:', X_vl.shape, 'Y_vl:', Y_vl.shape)

n_modes: 19
X_tr: (43, 400, 5) Y_tr: (43, 400, 19)
X_vl: (7, 400, 5) Y_vl: (7, 400, 19)


In [3]:
def make_hippo_legs(N):
    A = np.zeros((N, N))
    B = np.zeros((N, 1))
    for n in range(N):
        B[n, 0] = (2 * n + 1) ** 0.5
        for k in range(N):
            if n > k:
                A[n, k] = -(2 * n + 1) ** 0.5 * (2 * k + 1) ** 0.5
            elif n == k:
                A[n, k] = -(n + 1)
    return A, B


def get_activation(name):
    if name == 'relu':
        return nn.ReLU()
    if name == 'gelu':
        return nn.GELU()
    if name == 'silu':
        return nn.SiLU()
    if name == 'tanh':
        return nn.Tanh()
    raise ValueError(f'Unknown activation: {name}')


class S4Layer(nn.Module):
    def __init__(self, d_input, d_state=64, dropout=0.1, max_len=512):
        super().__init__()
        A_np, B_np = make_hippo_legs(d_state)
        step = 0.01
        I = np.eye(d_state)
        Ainv = np.linalg.solve(I - A_np * step / 2, I)
        A_bar = Ainv @ (I + A_np * step / 2)
        B_bar = (Ainv * step) @ B_np

        self.register_buffer('A_bar', torch.tensor(A_bar, dtype=torch.float32))
        self.register_buffer('B_bar', torch.tensor(B_bar.squeeze(), dtype=torch.float32))

        A_powers = np.zeros((max_len, d_state, d_state), dtype=np.float64)
        A_powers[0] = np.eye(d_state)
        for k in range(1, max_len):
            A_powers[k] = A_powers[k - 1] @ A_bar.astype(np.float64)
        self.register_buffer('A_powers', torch.tensor(A_powers, dtype=torch.float32))

        self.C = nn.Parameter(torch.randn(d_input, d_state) * 0.01)
        self.D = nn.Parameter(torch.randn(d_input) * 0.01)
        self.out_proj = nn.Linear(d_input, d_input)
        self.norm = nn.LayerNorm(d_input)
        self.dropout = nn.Dropout(dropout)

    def _kernel(self, L):
        return torch.einsum('hn,lnm,m->hl', self.C, self.A_powers[:L], self.B_bar)

    def forward(self, u):
        B, L, H = u.shape
        K = self._kernel(L)
        u_f = torch.fft.rfft(u.transpose(1, 2), n=2 * L)
        K_f = torch.fft.rfft(K, n=2 * L)
        y = torch.fft.irfft(u_f * K_f.unsqueeze(0), n=2 * L)[..., :L]
        y = y + u.transpose(1, 2) * self.D.unsqueeze(0).unsqueeze(-1)
        y = y.transpose(1, 2)
        y = self.dropout(F.gelu(self.out_proj(y)))
        return self.norm(y)


class S4Model(nn.Module):
    def __init__(self, n_sensors, n_modes, d_model=64, d_state=64, n_layers=2, dropout=0.1, max_len=512, activation='tanh'):
        super().__init__()
        self.input_proj = nn.Linear(n_sensors, d_model)
        self.layers = nn.ModuleList([S4Layer(d_model, d_state=d_state, dropout=dropout, max_len=max_len) for _ in range(n_layers)])
        self.decoder = nn.Sequential(nn.Linear(d_model, 128), get_activation(activation), nn.Linear(128, n_modes))

    def forward(self, x):
        h = self.input_proj(x)
        for layer in self.layers:
            h = h + layer(h)
        return self.decoder(h)


class S4DLayer(nn.Module):
    def __init__(self, d_input, d_state=64, dropout=0.1):
        super().__init__()
        A_np, _ = make_hippo_legs(d_state)
        eigs = np.linalg.eigvals(A_np)
        eigs = eigs[np.argsort(eigs.real)]

        self.log_neg_real = nn.Parameter(torch.log(-torch.tensor(eigs.real, dtype=torch.float32).clamp(max=-1e-4)))
        self.imag = nn.Parameter(torch.tensor(eigs.imag, dtype=torch.float32))

        self.B_re = nn.Parameter(torch.randn(d_input, d_state) * 0.01)
        self.B_im = nn.Parameter(torch.randn(d_input, d_state) * 0.01)
        self.C_re = nn.Parameter(torch.randn(d_input, d_state) * 0.01)
        self.C_im = nn.Parameter(torch.randn(d_input, d_state) * 0.01)
        self.D = nn.Parameter(torch.randn(d_input) * 0.01)
        self.log_step = nn.Parameter(torch.zeros(d_input).uniform_(-4, -1))
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_input)

    def _lambda(self):
        return -torch.exp(self.log_neg_real) + 1j * self.imag

    def forward(self, u):
        B, L, H = u.shape
        lam = self._lambda()
        step = torch.exp(self.log_step).clamp(min=1e-5, max=1.0).mean()
        lam_bar = torch.exp(lam * step)

        Bc = self.B_re + 1j * self.B_im
        Cc = self.C_re + 1j * self.C_im
        powers = lam_bar.unsqueeze(0).pow(torch.arange(L, device=u.device).unsqueeze(-1).float())
        K = torch.einsum('hn,ln,hn->hl', Cc, powers, Bc * step).real

        u_f = torch.fft.rfft(u.transpose(1, 2), n=2 * L)
        K_f = torch.fft.rfft(K, n=2 * L)
        y = torch.fft.irfft(u_f * K_f.unsqueeze(0), n=2 * L)[..., :L]
        y = y + u.transpose(1, 2) * self.D.unsqueeze(0).unsqueeze(-1)
        y = y.transpose(1, 2)
        y = self.dropout(F.gelu(y))
        return self.norm(y)


class S4DModel(nn.Module):
    def __init__(self, n_sensors, n_modes, d_model=64, d_state=64, n_layers=2, dropout=0.1, activation='tanh'):
        super().__init__()
        self.input_proj = nn.Linear(n_sensors, d_model)
        self.layers = nn.ModuleList([S4DLayer(d_model, d_state=d_state, dropout=dropout) for _ in range(n_layers)])
        self.decoder = nn.Sequential(nn.Linear(d_model, 128), get_activation(activation), nn.Linear(128, n_modes))

    def forward(self, x):
        h = self.input_proj(x)
        for layer in self.layers:
            h = h + layer(h)
        return self.decoder(h)


def make_model(model_name, **kwargs):
    if model_name == 'S4':
        return S4Model(**kwargs).to(device)
    if model_name == 'S4D':
        return S4DModel(**kwargs).to(device)
    raise ValueError(model_name)

In [4]:
def field_rmsre_pct(y_true_sc, y_pred_sc):
    n_traj, L, n_m = y_true_sc.shape
    yt = scaler.inverse_transform(y_true_sc.reshape(-1, n_m)).reshape(n_traj, L, n_m)
    yp = scaler.inverse_transform(y_pred_sc.reshape(-1, n_m)).reshape(n_traj, L, n_m)
    ft = yt @ Phi.T
    fp = yp @ Phi.T
    return 100.0 * np.sqrt(np.mean((fp - ft) ** 2)) / (np.sqrt(np.mean(ft ** 2)) + 1e-12)


def run_study(model_name, n_trials=30, epochs=80, study_suffix='full', show_progress=False, log_trial_params=True):
    def objective(trial):
        d_model = trial.suggest_categorical('d_model', [64, 96, 128, 256])
        d_state = trial.suggest_categorical('d_state', [32, 64, 96])
        n_layers = trial.suggest_int('n_layers', 1, 4)
        dropout = trial.suggest_float('dropout', 0.0, 0.3)
        activation = trial.suggest_categorical('activation', ['tanh', 'relu', 'gelu', 'silu'])
        lr = trial.suggest_float('lr', 1e-4, 3e-3, log=True)
        wd = trial.suggest_float('weight_decay', 1e-7, 1e-3, log=True)
        batch_size = trial.suggest_categorical('batch_size', [2, 4, 8, 16])

        trial_params = {
            'd_model': d_model,
            'd_state': d_state,
            'n_layers': n_layers,
            'dropout': dropout,
            'activation': activation,
            'lr': lr,
            'weight_decay': wd,
            'batch_size': batch_size,
        }
        if log_trial_params:
            print(f"[{model_name}] Trial {trial.number} params: {trial_params}")

        if model_name == 'S4':
            model = make_model(
                model_name,
                n_sensors=n_sensors,
                n_modes=n_modes,
                d_model=d_model,
                d_state=d_state,
                n_layers=n_layers,
                dropout=dropout,
                activation=activation,
                max_len=max(512, Nt),
            )
        elif model_name == 'S4D':
            model = make_model(
                model_name,
                n_sensors=n_sensors,
                n_modes=n_modes,
                d_model=d_model,
                d_state=d_state,
                n_layers=n_layers,
                dropout=dropout,
                activation=activation,
            )
        else:
            raise ValueError(model_name)

        tr_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(Y_tr))
        vl_ds = TensorDataset(torch.tensor(X_vl), torch.tensor(Y_vl))
        tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
        vl_dl = DataLoader(vl_ds, batch_size=batch_size)

        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        best_val = float('inf')

        with mlflow.start_run(run_name=f'{model_name}_trial_{trial.number}', nested=True):
            mlflow.log_params({
                'model': model_name,
                'trial': trial.number,
                **trial_params,
                'epochs': epochs,
            })

            for ep in range(epochs):
                model.train()
                tr_loss = 0.0
                tr_n = 0
                for xb, yb in tr_dl:
                    xb, yb = xb.to(device), yb.to(device)
                    pred = model(xb)
                    loss = F.mse_loss(pred, yb)
                    opt.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                    tr_loss += loss.item()
                    tr_n += 1

                model.eval()
                vl_loss = 0.0
                vl_n = 0
                with torch.no_grad():
                    for xb, yb in vl_dl:
                        xb, yb = xb.to(device), yb.to(device)
                        vl_loss += F.mse_loss(model(xb), yb).item()
                        vl_n += 1

                tr_mse = tr_loss / max(tr_n, 1)
                val_mse = vl_loss / max(vl_n, 1)
                best_val = min(best_val, val_mse)

                mlflow.log_metric('train_mse', tr_mse, step=ep)
                mlflow.log_metric('val_mse', val_mse, step=ep)

                trial.report(val_mse, ep)
                if trial.should_prune():
                    raise optuna.TrialPruned()

            with torch.no_grad():
                y_pred_vl = model(torch.tensor(X_vl, dtype=torch.float32, device=device)).cpu().numpy()
            rmsre = field_rmsre_pct(Y_vl, y_pred_vl)

            mlflow.log_metric('best_val_mse', best_val)
            mlflow.log_metric('val_field_rmsre_pct', rmsre)

        trial.set_user_attr('val_field_rmsre_pct', float(rmsre))
        return best_val

    study = optuna.create_study(
        study_name=f'transport_{model_name.lower()}_{study_suffix}',
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=0),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=max(5, epochs // 8)),
    )

    with mlflow.start_run(run_name=f'{model_name}_{study.study_name}_controller'):
        mlflow.log_params({'model': model_name, 'n_trials': n_trials, 'epochs': epochs, 'log_trial_params': log_trial_params})
        study.optimize(objective, n_trials=n_trials, show_progress_bar=show_progress)
        mlflow.log_metric('study_best_val_mse', study.best_value)
        mlflow.log_metric('study_best_val_field_rmsre_pct', study.best_trial.user_attrs.get('val_field_rmsre_pct', np.nan))
        for k, v in study.best_params.items():
            mlflow.log_param(f'best_{k}', v)

    return study


mlflow.set_tracking_uri('file:./mlruns')
mlflow.set_experiment('transport_ssm_optuna')
print('MLflow URI:', mlflow.get_tracking_uri())

MLflow URI: file:./mlruns


## Smoke Test
Run tiny studies first to verify Optuna + MLflow + reconstruction metric logging.

In [5]:
smoke_s4 = run_study('S4', n_trials=2, epochs=5, study_suffix='smoke')
smoke_s4d = run_study('S4D', n_trials=2, epochs=5, study_suffix='smoke')

print('Smoke complete')
print('S4   best POD val MSE:', f'{smoke_s4.best_value:.4e}', '| field RMSRE%:', f"{smoke_s4.best_trial.user_attrs['val_field_rmsre_pct']:.3f}")
print('S4D  best POD val MSE:', f'{smoke_s4d.best_value:.4e}', '| field RMSRE%:', f"{smoke_s4d.best_trial.user_attrs['val_field_rmsre_pct']:.3f}")

[S4] Trial 0 params: {'d_model': 96, 'd_state': 64, 'n_layers': 4, 'dropout': 0.2890988281503088, 'activation': 'relu', 'lr': 0.002329262676497441, 'weight_decay': 1.923730509654648e-07, 'batch_size': 8}
[S4] Trial 1 params: {'d_model': 96, 'd_state': 32, 'n_layers': 1, 'dropout': 0.28340067511487516, 'activation': 'silu', 'lr': 0.00047183412869566564, 'weight_decay': 1.878173875716189e-05, 'batch_size': 4}
[S4D] Trial 0 params: {'d_model': 96, 'd_state': 64, 'n_layers': 4, 'dropout': 0.2890988281503088, 'activation': 'relu', 'lr': 0.002329262676497441, 'weight_decay': 1.923730509654648e-07, 'batch_size': 8}
[S4D] Trial 1 params: {'d_model': 96, 'd_state': 32, 'n_layers': 1, 'dropout': 0.28340067511487516, 'activation': 'silu', 'lr': 0.00047183412869566564, 'weight_decay': 1.878173875716189e-05, 'batch_size': 4}
Smoke complete
S4   best POD val MSE: 5.4553e-01 | field RMSRE%: 35.242
S4D  best POD val MSE: 4.7564e-01 | field RMSRE%: 31.207


## Full Study

Thorough search budget:
- `S4`: 60 trials, 100 epochs
- `S4D`: 80 trials, 100 epochs

This can take a while, but gives much better coverage of the search space.

In [6]:
# Thorough search budget
BUDGET_S4_TRIALS = 10
BUDGET_S4D_TRIALS = 10
BUDGET_EPOCHS = 500

full_s4 = run_study(
    'S4',
    n_trials=BUDGET_S4_TRIALS,
    epochs=BUDGET_EPOCHS,
    study_suffix='full_thorough',
    show_progress=True,
    log_trial_params=True,
 )
full_s4d = run_study(
    'S4D',
    n_trials=BUDGET_S4D_TRIALS,
    epochs=BUDGET_EPOCHS,
    study_suffix='full_thorough',
    show_progress=True,
    log_trial_params=True,
 )

rows = [
    {
        'model': 'S4',
        'best_val_pod_mse': full_s4.best_value,
        'best_val_field_rmsre_pct': full_s4.best_trial.user_attrs['val_field_rmsre_pct'],
        **full_s4.best_params,
    },
    {
        'model': 'S4D',
        'best_val_pod_mse': full_s4d.best_value,
        'best_val_field_rmsre_pct': full_s4d.best_trial.user_attrs['val_field_rmsre_pct'],
        **full_s4d.best_params,
    },
]

display(pd.DataFrame(rows))

  0%|          | 0/20 [00:00<?, ?it/s]

[S4] Trial 0 params: {'d_model': 96, 'd_state': 64, 'n_layers': 4, 'dropout': 0.2890988281503088, 'activation': 'relu', 'lr': 0.002329262676497441, 'weight_decay': 1.923730509654648e-07, 'batch_size': 8}


Best trial: 0. Best value: 0.250407:   5%|▌         | 1/20 [00:39<12:33, 39.64s/it]

[S4] Trial 1 params: {'d_model': 96, 'd_state': 32, 'n_layers': 1, 'dropout': 0.28340067511487516, 'activation': 'silu', 'lr': 0.00047183412869566564, 'weight_decay': 1.878173875716189e-05, 'batch_size': 4}


Best trial: 0. Best value: 0.250407:  10%|█         | 2/20 [01:01<08:47, 29.32s/it]

...

[S4D] Trial 18 params: {'d_model': 64, 'd_state': 96, 'n_layers': 2, 'dropout': 0.1432624132341657, 'activation': 'gelu', 'lr': 0.002916428232831814, 'weight_decay': 4.839278336457889e-07, 'batch_size': 2}


Best trial: 12. Best value: 0.283039:  95%|█████████▌| 19/20 [14:51<00:39, 39.88s/it]

[S4D] Trial 19 params: {'d_model': 96, 'd_state': 96, 'n_layers': 1, 'dropout': 0.0357170305744688, 'activation': 'tanh', 'lr': 0.0014379894162200651, 'weight_decay': 0.00012306265841911207, 'batch_size': 2}


Best trial: 12. Best value: 0.283039: 100%|██████████| 20/20 [16:13<00:00, 48.69s/it]


,model,best_val_pod_mse,best_val_field_rmsre_pct,d_model,d_state,n_layers,dropout,activation,lr,weight_decay,batch_size
0,S4,0.208539,16.708115,96,96,4,0.207632,relu,0.000769,0.000097,2
1,S4D,0.283039,NaN,96,96,2,0.067678,gelu,0.001032,0.000002,2
